# 01 - Prepare Flickr30K Metadata

## 1. Import libraries and define paths

In [1]:
from pathlib import Path
import json

import pandas as pd

# Dinh nghia duong dan tu notebook ve root project.
RAW_DIR = Path("../data/raw")
IMAGE_DIR = RAW_DIR / "Images"
CAPTION_PATH = RAW_DIR / "captions.txt"
PROCESSED_DIR = Path("../data/processed")
OUTPUT_PATH = PROCESSED_DIR / "metadata.json"

## 2. Check input files

In [2]:
# Kiem tra du lieu dau vao truoc khi xu ly.
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Khong tim thay thu muc anh: {IMAGE_DIR}")

if not CAPTION_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay file captions: {CAPTION_PATH}")

print(f"Image directory: {IMAGE_DIR}")
print(f"Caption file: {CAPTION_PATH}")

Image directory: ..\data\raw\Images
Caption file: ..\data\raw\captions.txt


## 3. Load captions

In [3]:
# Doc CSV bang pandas, khong tu tach caption bang dau phay.
df = pd.read_csv(CAPTION_PATH)

display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

,image,caption
0,1000092795.jpg,Two young guys with shaggy hair look at their...
1,1000092795.jpg,"Two young , White males are outside near many..."
2,1000092795.jpg,Two men in green shirts are standing in a yard .
3,1000092795.jpg,A man in a blue shirt standing in a garden .
4,1000092795.jpg,Two friends enjoy time spent together .


Shape: (158915, 2)
Columns: ['image', 'caption']


## 4. Validate caption columns

In [4]:
required_columns = {"image", "caption"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"File captions thieu cot bat buoc: {sorted(missing_columns)}")

print("Caption columns hop le.")

Caption columns hop le.


## 5. Clean captions

In [5]:
initial_rows = len(df)

# Loai bo dong thieu image hoac caption.
df_clean = df.dropna(subset=["image", "caption"]).copy()
df_clean["image"] = df_clean["image"].astype(str).str.strip()
df_clean["caption"] = df_clean["caption"].astype(str).str.strip()

print(f"So dong ban dau: {initial_rows}")
print(f"So dong sau khi drop missing: {len(df_clean)}")

So dong ban dau: 158915
So dong sau khi drop missing: 158914


## 6. Keep only images with exactly 5 captions

In [6]:
caption_counts = df_clean.groupby("image")["caption"].size()
valid_images = caption_counts[caption_counts == 5].index

df_filtered = df_clean[df_clean["image"].isin(valid_images)].copy()

print(f"So anh ban dau: {caption_counts.shape[0]}")
print(f"So anh hop le co dung 5 caption: {len(valid_images)}")
print(f"So dong sau khi loc: {len(df_filtered)}")

So anh ban dau: 31783
So anh hop le co dung 5 caption: 31782
So dong sau khi loc: 158910


## 7. Check image files

In [7]:
# Lay danh sach file anh that trong thu muc raw.
image_files = {path.name for path in IMAGE_DIR.iterdir() if path.is_file()}
caption_images = set(df_filtered["image"].unique())

missing_images = sorted(caption_images - image_files)
extra_images = sorted(image_files - caption_images)

print(f"So anh co trong caption nhung khong co file: {len(missing_images)}")
print(f"So file anh thua khong co caption: {len(extra_images)}")

if missing_images:
    raise FileNotFoundError(
        "Co image id trong captions nhung khong tim thay file anh. "
        f"Vi du: {missing_images[:5]}"
    )

if extra_images:
    print(f"Warning: co file anh thua khong co caption. Vi du: {extra_images[:5]}")

So anh co trong caption nhung khong co file: 0
So file anh thua khong co caption: 1


## 8. Build metadata

In [8]:
metadata = []

# Luu image_path tuong doi theo root project de notebook/app sau dung truc tiep.
for image_id, group in df_filtered.groupby("image", sort=True):
    captions = group["caption"].tolist()
    metadata.append(
        {
            "image_id": image_id,
            "image_path": f"data/raw/Images/{image_id}",
            "captions": captions,
        }
    )

print(f"So item metadata: {len(metadata)}")
metadata[0] if metadata else None

So item metadata: 31782


{'image_id': '1000092795.jpg',
 'image_path': 'data/raw/Images/1000092795.jpg',
 'captions': ['Two young guys with shaggy hair look at their hands while hanging out in the yard .',
  'Two young , White males are outside near many bushes .',
  'Two men in green shirts are standing in a yard .',
  'A man in a blue shirt standing in a garden .',
  'Two friends enjoy time spent together .']}

## 9. Save metadata

In [9]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Da luu metadata tai: {OUTPUT_PATH}")
print(f"So item metadata: {len(metadata)}")

Da luu metadata tai: ..\data\processed\metadata.json
So item metadata: 31782


## 10. Sanity check

In [10]:
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print(loaded_metadata[0])

required_keys = {"image_id", "image_path", "captions"}

for idx, item in enumerate(loaded_metadata):
    missing_keys = required_keys - set(item.keys())
    if missing_keys:
        raise ValueError(f"Item {idx} thieu keys: {sorted(missing_keys)}")
    if len(item["captions"]) != 5:
        raise ValueError(f"Item {idx} khong co dung 5 captions: {item['image_id']}")

print("Sanity check passed.")

{'image_id': '1000092795.jpg', 'image_path': 'data/raw/Images/1000092795.jpg', 'captions': ['Two young guys with shaggy hair look at their hands while hanging out in the yard .', 'Two young , White males are outside near many bushes .', 'Two men in green shirts are standing in a yard .', 'A man in a blue shirt standing in a garden .', 'Two friends enjoy time spent together .']}
Sanity check passed.
